# PolarExpress — Polynomial Composition Visualiser

Reproduces **Figure 2** from *The Polar Express: Optimal Matrix Sign Methods and Their Application to the Muon Algorithm* ([arXiv:2505.16932](https://arxiv.org/abs/2505.16932)) and provides additional convergence diagnostics.

All polynomial coefficients are imported from the `polar_express` package via `optimal_composition`; no Remez algorithm is re-implemented here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colormaps
import matplotlib

# All coefficient computation is imported from the package — no Remez here.
from polar_express import optimal_composition

matplotlib.rcParams.update({
    "font.family":       "serif",
    "mathtext.fontset":  "cm",
    "font.size":         12,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "legend.fontsize":   11,
    "lines.linewidth":   2.5,
    "axes.linewidth":    1.2,
    "axes.grid":         True,
    "grid.linestyle":    "--",
    "grid.linewidth":    0.6,
    "grid.alpha":        0.35,
    "legend.framealpha": 0.95,
    "figure.dpi":        120,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

def eval_poly(coeffs, x):
    """Evaluate odd polynomial  c0·x + c1·x³ + …  at scalar or array x."""
    return sum(c * x**(2*i + 1) for i, c in enumerate(coeffs))

## Figure 2 — Polynomial composition, step by step

Each subplot shows one polynomial $p_t(x)$, computed by `optimal_composition` (Remez algorithm in the package), to approximate 1 on $[\ell_t, u_t]$.  After applying $p_t$, singular values are mapped to the tighter interval $[\ell_{t+1}, u_{t+1}]$ where $\ell_{t+1} = p_t(\ell_t)$ and $u_{t+1} = 2 - \ell_{t+1}$.

Dashed lines trace the input-bounds rectangle; the red dot marks $(\ell_t,\, \ell_{t+1})$.

In [ ]:
def plot_figure2(l1, n_steps, degree=5, figsize=None):
    """
    Reproduce Figure 2 from the paper.

    Parameters
    ----------
    l1 : float   Initial lower bound on singular values.
    n_steps : int   Number of polynomial steps (= number of subplot panels).
    degree : {3, 5}   Polynomial degree.
    """
    coeffs_seq = optimal_composition(
        l1, n_steps, degree=degree, safety_factor_eps=0.01, cushion=0.02
    )
    fig, axes = plt.subplots(
        1, n_steps, figsize=figsize or (5 * n_steps, 5), sharey=True
    )
    if n_steps == 1:
        axes = [axes]

    cmap   = colormaps["viridis"]
    x_full = np.linspace(0, 2, 1000)
    l_cur, u_cur = float(l1), 1.0

    for i, coeffs in enumerate(coeffs_seq):
        color  = cmap(i / max(1, n_steps - 1))
        ax     = axes[i]
        l_next = float(eval_poly(coeffs, l_cur))
        u_next = 2.0 - l_next

        # Polynomial curve and y = 1 reference
        ax.plot(x_full, eval_poly(coeffs, x_full), color=color, linewidth=3.5,
                label=r'$p_{}$'.format(i + 1))
        ax.plot(x_full, np.ones_like(x_full), 'k-', linewidth=2.5, alpha=0.85)

        # Dashed lines tracing the bounds rectangle
        ax.plot([l_cur, l_cur], [0,      u_next], 'k--', linewidth=1.2)
        ax.plot([u_cur, u_cur], [0,      u_next], 'k--', linewidth=1.2)
        ax.plot([0,     u_cur], [l_next, l_next], 'k--', linewidth=1.2)
        ax.plot([0,     u_cur], [u_next, u_next], 'k--', linewidth=1.2)

        # Shaded rectangle: input interval → output interval
        ax.add_patch(patches.Rectangle(
            (l_cur, l_next), u_cur - l_cur, u_next - l_next,
            linewidth=1, edgecolor='none', facecolor=color, alpha=0.2))

        # Bound labels
        ax.text(l_cur, -0.15, r'$\ell_{}$'.format(i + 1), fontsize=14, ha='center', va='top')
        ax.text(u_cur, -0.15, r'$u_{}$'.format(i + 1),    fontsize=14, ha='center', va='top')
        if i == 0:
            ax.text(-0.15, l_next, r'$\ell_{}$'.format(i + 2), fontsize=14, ha='right', va='center')
            ax.text(-0.15, u_next, r'$u_{}$'.format(i + 2),    fontsize=14, ha='right', va='center')
            ax.set_yticks([0, 2])

        # Red dot at (ℓ_t, ℓ_{t+1})
        ax.scatter([l_cur], [l_next], color='red', s=50, zorder=5)

        ax.set_xlim(0, 2);  ax.set_ylim(0, 2)
        ax.set_xticks([0, 2])
        ax.legend(loc='upper right', fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=12)

        l_cur, u_cur = l_next, u_next

    deg_name = {3: 'cubic', 5: 'quintic'}[degree]
    fig.suptitle(
        r'First {} optimal {} polynomials  ($\ell_1 = {}$)'.format(n_steps, deg_name, l1),
        fontsize=14, y=1.02)
    plt.tight_layout()
    return fig, axes

In [ ]:
# Paper Figure 2: degree=5, ℓ₁=0.03, 3 steps
fig, _ = plot_figure2(l1=0.03, n_steps=3, degree=5)
plt.show()

In [ ]:
# Compare both polynomial degrees (same ℓ₁ and n_steps)
for degree in [3, 5]:
    fig, _ = plot_figure2(l1=0.03, n_steps=3, degree=degree)
    plt.show()

## Bounds evolution

Shows how $[\ell_t, u_t]$ contracts toward 1 as the polynomial composition proceeds, for each polynomial degree.  Higher degree shrinks the interval faster (at the cost of one extra matrix multiply per step).

In [ ]:
def plot_bounds_evolution(l1, n_steps=10, degrees=(3, 5), figsize=(9, 4.5)):
    """
    Plot ℓ_t and u_t vs iteration t for each polynomial degree.

    Uses optimal_composition from the package for each degree.
    """
    fig, ax = plt.subplots(figsize=figsize)
    palette = {3: '#ff7f0e', 5: '#1f77b4'}
    styles  = {3: '-.', 5: '-'}

    for degree in degrees:
        coeffs_seq = optimal_composition(
            l1, n_steps, degree=degree, safety_factor_eps=0.01, cushion=0.02
        )
        ls, us = [float(l1)], [1.0]
        l_cur = float(l1)
        for coeffs in coeffs_seq:
            l_cur = float(eval_poly(coeffs, l_cur))
            ls.append(l_cur)
            us.append(2.0 - l_cur)

        t   = np.arange(n_steps + 1)
        col = palette[degree]
        sty = styles[degree]
        ax.plot(t, ls, color=col, linestyle=sty, linewidth=2.5,
                marker='o', markersize=5, label=r'$\ell_t$  (d={})'.format(degree))
        ax.plot(t, us, color=col, linestyle=sty, linewidth=2.5,
                marker='s', markersize=5, alpha=0.55,
                label=r'$u_t$  (d={})'.format(degree))
        ax.fill_between(t, ls, us, color=col, alpha=0.07)

    ax.axhline(1.0, color='k', linestyle=':', linewidth=1.5, label='$y = 1$')
    ax.set_xlabel('Iteration $t$')
    ax.set_ylabel(r'$\ell_t$  and  $u_t$')
    ax.set_title(
        r'Bounds $[\ell_t, u_t]$ contracting toward 1  ($\ell_1 = {}$)'.format(l1),
        fontsize=13)
    ax.legend(ncol=len(degrees), fontsize=10, loc='center right')
    plt.tight_layout()
    return fig

fig = plot_bounds_evolution(l1=0.001, n_steps=10)
plt.show()

## Method comparison — PolarExpress vs fixed polynomials

Compares `PolarExpress (d=5)` (adaptive composition from the package) against three fixed-coefficient methods from the literature:

| Method | Degree | Coefficients |
|--------|--------|-------------|
| Newton-Schulz | 3 | $1.5x - 0.5x^3$ (fixed) |
| Keller (Jordan) | 5 | $3.4445x - 4.775x^3 + 2.0315x^5$ (fixed) |
| Jiacheng | 5 | 6-polynomial hand-crafted sequence (fixed) |

**Left panel**: composition applied to $x \in [0,1]$ after $T$ steps.  
**Right panel**: $\max_{x}|p^T(x) - 1|$ vs iteration $T$.

In [ ]:
# ── Fixed-coefficient comparison polynomials ─────────────────────────────────
# These are not part of the polar_express package; they are hardcoded here
# for comparison purposes.

_NS_COEFFS      = (1.5, -0.5)                   # Newton-Schulz (degree 3, fixed)
_KELLER_COEFFS  = (3.4445, -4.7750, 2.0315)      # Keller / Jordan (degree 5, fixed)
_JIACHENG_LIST  = [                              # Jiacheng's 6-step sequence
    (3955/1024, -8306/1024, 5008/1024),
    (3735/1024, -6681/1024, 3463/1024),
    (3799/1024, -6499/1024, 3211/1024),
    (4019/1024, -6385/1024, 2906/1024),
    (2677/1024, -3029/1024, 1162/1024),
    (2172/1024, -1833/1024,  682/1024),
]


def method_comparison(l1=0.001, n_steps=12, n_stop=6):
    """
    Two-panel comparison: polynomial output after n_stop steps (left)
    and max-error convergence to 1 over all steps (right).
    """
    x_grid = np.linspace(0, 1, 500)

    # ── PolarExpress (d=5, adaptive) ────────────────────────────────────────
    pe5_coeffs = optimal_composition(
        l1, n_steps, degree=5, safety_factor_eps=0.01, cushion=0.02
    )
    pe5_x    = x_grid.copy()
    pe5_errs = [np.linalg.norm(pe5_x[1:] - 1, np.inf)]
    pe5_snap = None
    for k in range(n_steps):
        c     = pe5_coeffs[k] if k < len(pe5_coeffs) else pe5_coeffs[-1]
        pe5_x = eval_poly(c, pe5_x)
        pe5_errs.append(np.linalg.norm(pe5_x[1:] - 1, np.inf))
        if k + 1 == n_stop:
            pe5_snap = pe5_x.copy()

    # ── Newton-Schulz (fixed polynomial) ────────────────────────────────────
    ns_x    = x_grid.copy()
    ns_errs = [np.linalg.norm(ns_x[1:] - 1, np.inf)]
    ns_snap = None
    for k in range(n_steps):
        ns_x = eval_poly(_NS_COEFFS, ns_x)
        ns_errs.append(np.linalg.norm(ns_x[1:] - 1, np.inf))
        if k + 1 == n_stop:
            ns_snap = ns_x.copy()

    # ── Keller / Jordan (fixed polynomial) ──────────────────────────────────
    kl_x    = x_grid.copy()
    kl_errs = [np.linalg.norm(kl_x[1:] - 1, np.inf)]
    kl_snap = None
    for k in range(n_steps):
        kl_x = eval_poly(_KELLER_COEFFS, kl_x)
        kl_errs.append(np.linalg.norm(kl_x[1:] - 1, np.inf))
        if k + 1 == n_stop:
            kl_snap = kl_x.copy()

    # ── Jiacheng (fixed sequence) ────────────────────────────────────────────
    ji_x    = x_grid.copy()
    ji_errs = [np.linalg.norm(ji_x[1:] - 1, np.inf)]
    ji_snap = None
    for k in range(n_steps):
        c     = _JIACHENG_LIST[k] if k < len(_JIACHENG_LIST) else _JIACHENG_LIST[-1]
        ji_x  = eval_poly(c, ji_x)
        ji_errs.append(np.linalg.norm(ji_x[1:] - 1, np.inf))
        if k + 1 == n_stop:
            ji_snap = ji_x.copy()

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    iters = np.arange(n_steps + 1)

    kw = dict(linewidth=4, alpha=0.85)
    ax1.plot(x_grid, pe5_snap, color='k',       **kw, label=r'PolarExpress ($\ell={}$)'.format(l1))
    ax1.plot(x_grid, ji_snap,  color='#8A2BE2', **kw, linestyle=':',  label='Jiacheng')
    ax1.plot(x_grid, ns_snap,  color='#228B22', **kw, linestyle='--', label='Newton-Schulz')
    ax1.plot(x_grid, kl_snap,  color='#FF0000', **kw, linestyle='-.', label='Keller (Jordan)')
    ax1.axvline(l1, color='gray', linestyle='--', linewidth=1.5, label=r'$\ell_1$')
    ax1.set_xlabel(r'$x$',           fontsize=20)
    ax1.set_ylabel(r'$p^T(x)$',      fontsize=20)
    ax1.tick_params(axis='both', labelsize=16)
    ax1.set_title('After {} iterations'.format(n_stop), fontsize=16)

    ax2.semilogy(iters, pe5_errs, color='k',       **kw, label=r'PolarExpress ($\ell={}$)'.format(l1))
    ax2.semilogy(iters, ji_errs,  color='#8A2BE2', **kw, linestyle=':',  label='Jiacheng')
    ax2.semilogy(iters, ns_errs,  color='#228B22', **kw, linestyle='--', label='Newton-Schulz')
    ax2.semilogy(iters, kl_errs,  color='#FF0000', **kw, linestyle='-.', label='Keller (Jordan)')
    ax2.set_xlabel(r'Iteration ($T$)',            fontsize=20)
    ax2.set_ylabel(r'$\max_x |p^T(x) - 1|$',     fontsize=20)
    ax2.tick_params(axis='both', labelsize=16)
    ax2.set_title('Error convergence', fontsize=16)

    handles, labels = ax1.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(by_label.values(), by_label.keys(),
               loc='lower center', bbox_to_anchor=(0.5, -0.08),
               ncol=len(by_label), fontsize=15, fancybox=True)
    plt.tight_layout()
    return fig


fig = method_comparison(l1=0.001, n_steps=12, n_stop=6)
plt.show()

## Composition arrows

Viridis colour-graded view of the full polynomial sequence.  Arrows trace the lower bound $\ell_t$ as it is lifted toward 1 by each successive polynomial.

In [ ]:
def plot_composition_arrows(l1, n_steps=12, degree=5, figsize=(16, 4.8)):
    """
    Viridis colour-graded sequence of polynomials with arrows that trace
    the evolution of the lower bound ℓ_t → ℓ_{t+1}.

    Uses optimal_composition from the package.
    """
    coeffs_seq = optimal_composition(
        l1, n_steps, degree=degree, safety_factor_eps=0.01, cushion=0.02
    )
    fig, ax = plt.subplots(figsize=figsize)
    cmap   = colormaps["viridis"]
    x_grid = np.linspace(0, 1, 500)
    l_cur  = float(l1)

    for i, coeffs in enumerate(coeffs_seq):
        color  = cmap(i / max(1, len(coeffs_seq) - 1))
        l_next = float(eval_poly(coeffs, l_cur))

        ax.plot(x_grid, eval_poly(coeffs, x_grid), color=color, linewidth=3.5,
                alpha=0.85, label=r'$\ell={:.3f}$'.format(l_cur))

        # Horizontal arrow: (ℓ_cur, ℓ_next) → (ℓ_next, ℓ_next)
        ax.annotate('', xy=(l_next, l_next), xytext=(l_cur, l_next),
                    arrowprops=dict(arrowstyle='->'))
        # Vertical arrow: (ℓ_cur, ℓ_cur) → (ℓ_cur, ℓ_next)
        if i > 0:
            ax.annotate('', xy=(l_cur, l_next), xytext=(l_cur, l_cur),
                        arrowprops=dict(arrowstyle='->'))

        ax.scatter([l_cur], [l_next], color='red', s=50, zorder=5)
        l_cur = l_next

    ax.plot(x_grid, x_grid, 'k--', linewidth=0.5)
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_xlabel(r'$x$',    fontsize=14)
    ax.set_ylabel(r'$p(x)$', fontsize=14)
    ax.legend(loc='upper left', fontsize=11)
    ax.tick_params(axis='both', labelsize=12)
    plt.tight_layout()
    return fig

fig = plot_composition_arrows(l1=0.001, n_steps=12, degree=5)
plt.show()